# 10 — Exemplos adversariais

Reproduzimos o experimento adversarial central de Wang et al. (arXiv:2211.00593, Appendix L) e de `Easy-Transformer/easy_transformer/advex.py` no commit histórico `ea15315dd24481e9e2ac5c3ef335d82907a1dc34`.

Para cada prompt IOI, inserimos no meio do texto duas cláusulas naturais, cada uma contendo o IO. Isso cria duas ocorrências extras do IO sem mudar IO, S ou a resposta. O par baseline/adversarial compartilha exatamente o mesmo exemplo-base; todas as posições do adversarial são recalculadas depois da inserção, sem reaproveitar índices deslocados do baseline.

In [1]:
from datetime import datetime, timezone
import hashlib
from importlib.metadata import version
import json
from pathlib import Path
import subprocess

import pandas as pd
import plotly.express as px
import torch

from ioi_replication.paper import (
    DOUBLE_ADVERSARIAL_TEMPLATES,
    EASY_TRANSFORMER_ADVEX_URL,
    EASY_TRANSFORMER_SOURCE_COMMIT,
    PAPER_ADVERSARIAL_TARGETS,
    build_adversarial_dataset,
    io_probability,
    logit_difference,
    summarize_adversarial_values,
)
from ioi_replication.runtime import load_gpt2_small, select_device

SEED = 510
N = 500
BATCH_SIZE = 16
MODEL_NAME = "gpt2"
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
device = select_device()
print(f"Dispositivo: {device}; seed: {SEED}; N: {N}; batch: {BATCH_SIZE}")

Dispositivo: cuda; seed: 510; N: 500; batch: 16


## Fonte e amostragem

`advex.py` define cinco cláusulas e forma os 25 pares ordenados (`DOUBLE_ADX_TEMPLATE`), sorteando uniformemente um par por prompt com `random.choice`. Preservamos os textos, a ordem do produto cartesiano, os placeholders `[A]` e a inserção após a pontuação intermediária. A adaptação local torna esse sorteio determinístico com `random.Random(SEED)`.

O paper reporta, no adversarial, IO−S ≈ 1.23 e S>IO em 23.4% dos exemplos. Esses valores são alvos externos, não critérios de aprovação exata: versão do runtime, amostra, seed e gerador do dataset podem mudar os números observados.

In [2]:
model = load_gpt2_small(device)
dataset = build_adversarial_dataset(
    model.tokenizer,
    n=N,
    seed=SEED,
    pattern="mixed",
)

assert len(DOUBLE_ADVERSARIAL_TEMPLATES) == 25
assert dataset.prepend_bos is False
assert model.tokenizer.padding_side == "right"
assert torch.equal(
    dataset.base_positions["END"],
    dataset.base_attention_mask.sum(dim=1) - 2,
)
assert torch.equal(
    dataset.adversarial_positions["END"],
    dataset.adversarial_attention_mask.sum(dim=1) - 2,
)
assert torch.all(
    dataset.extra_io_positions
    < dataset.adversarial_positions["S2"][:, None]
)

for index in range(2):
    base_length = int(dataset.base_attention_mask[index].sum())
    adv_length = int(dataset.adversarial_attention_mask[index].sum())
    print(f"\nExemplo pareado {index}")
    print("baseline:", dataset.base_prompts[index])
    print("adversarial:", dataset.adversarial_prompts[index])
    print("tokens baseline:", model.tokenizer.batch_decode(
        dataset.base_tokens[index, :base_length]
    ))
    print("tokens adversarial:", model.tokenizer.batch_decode(
        dataset.adversarial_tokens[index, :adv_length]
    ))
    print("posições baseline:", {
        key: int(value[index]) for key, value in dataset.base_positions.items()
    })
    print("posições adversarial:", {
        key: int(value[index])
        for key, value in dataset.adversarial_positions.items()
    })
    print("IOs extras:", dataset.extra_io_positions[index].tolist())

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Exemplo pareado 0
baseline: After the lunch, Bradley and Benjamin went to the restaurant. Benjamin gave a necklace to Bradley
adversarial: After the lunch, Bradley and Benjamin went to the restaurant. Bradley was an enthusiast person. Bradley was enjoying the situation. Benjamin gave a necklace to Bradley
tokens baseline: ['After the lunch, Bradley and Benjamin went to the restaurant. Benjamin gave a necklace to Bradley']
tokens adversarial: ['After the lunch, Bradley and Benjamin went to the restaurant. Bradley was an enthusiast person. Bradley was enjoying the situation. Benjamin gave a necklace to Bradley']
posições baseline: {'IO': 4, 'S1': 6, 'S2': 12, 'S1+1': 7, 'END': 16}
posições adversarial: {'IO': 4, 'S1': 6, 'S2': 24, 'S1+1': 7, 'END': 28}
IOs extras: [12, 18]

Exemplo pareado 1
baseline: Friends William and Angela found a drink at the hospital. William gave it to Angela
adversarial: Friends William and Angela found a drink at the hospital. Angela enjoyed being with a frien

In [3]:
def evaluate_arm(tokens: torch.Tensor, end_positions: torch.Tensor):
    differences = []
    probabilities = []
    with torch.inference_mode():
        for start in range(0, len(dataset), BATCH_SIZE):
            stop = min(start + BATCH_SIZE, len(dataset))
            logits = model(tokens[start:stop].to(device))
            differences.append(
                logit_difference(
                    logits,
                    dataset.io_token_ids[start:stop].to(device),
                    dataset.s_token_ids[start:stop].to(device),
                    end_positions[start:stop].to(device),
                    reduction="none",
                ).cpu()
            )
            probabilities.append(
                io_probability(
                    logits,
                    dataset.io_token_ids[start:stop].to(device),
                    end_positions[start:stop].to(device),
                    reduction="none",
                ).cpu()
            )
            del logits
    return torch.cat(differences), torch.cat(probabilities)


baseline_ld, baseline_io_probability = evaluate_arm(
    dataset.base_tokens,
    dataset.base_positions["END"],
)
adversarial_ld, adversarial_io_probability = evaluate_arm(
    dataset.adversarial_tokens,
    dataset.adversarial_positions["END"],
)
metrics = summarize_adversarial_values(
    baseline_logit_difference=baseline_ld,
    adversarial_logit_difference=adversarial_ld,
    baseline_io_probability=baseline_io_probability,
    adversarial_io_probability=adversarial_io_probability,
)
print("baseline IO−S:", metrics.logit_difference.baseline.mean)
print("adversarial IO−S:", metrics.logit_difference.adversarial.mean)
print("baseline S>IO:", metrics.s_wins_error_rate.baseline.mean)
print("adversarial S>IO:", metrics.s_wins_error_rate.adversarial.mean)

baseline IO−S: 3.4282398223876953
adversarial IO−S: 0.4849517345428467
baseline S>IO: 0.0020000000949949026
adversarial S>IO: 0.4020000100135803


In [4]:
def estimate_dict(estimate):
    return {
        "mean": estimate.mean,
        "standard_error": estimate.standard_error,
        "ci95": list(estimate.ci95),
    }


def paired_dict(paired):
    return {
        "baseline": estimate_dict(paired.baseline),
        "adversarial": estimate_dict(paired.adversarial),
        "delta": estimate_dict(paired.delta),
    }


metrics_json = {
    "logit_difference_io_minus_s": paired_dict(metrics.logit_difference),
    "s_wins_error_rate": paired_dict(metrics.s_wins_error_rate),
    "io_probability": paired_dict(metrics.io_probability),
}
comparison = pd.DataFrame(
    [
        {
            "métrica": "IO−S",
            "baseline": metrics.logit_difference.baseline.mean,
            "adversarial": metrics.logit_difference.adversarial.mean,
            "paper adversarial": PAPER_ADVERSARIAL_TARGETS[
                "adversarial_logit_difference"
            ],
            "observado − paper": (
                metrics.logit_difference.adversarial.mean
                - PAPER_ADVERSARIAL_TARGETS[
                    "adversarial_logit_difference"
                ]
            ),
        },
        {
            "métrica": "S>IO (erro)",
            "baseline": metrics.s_wins_error_rate.baseline.mean,
            "adversarial": metrics.s_wins_error_rate.adversarial.mean,
            "paper adversarial": PAPER_ADVERSARIAL_TARGETS[
                "adversarial_s_wins_error_rate"
            ],
            "observado − paper": (
                metrics.s_wins_error_rate.adversarial.mean
                - PAPER_ADVERSARIAL_TARGETS[
                    "adversarial_s_wins_error_rate"
                ]
            ),
        },
        {
            "métrica": "P(IO)",
            "baseline": metrics.io_probability.baseline.mean,
            "adversarial": metrics.io_probability.adversarial.mean,
            "paper adversarial": float("nan"),
            "observado − paper": float("nan"),
        },
    ]
)
comparison

,métrica,baseline,adversarial,paper adversarial,observado − paper
0,IO−S,3.428240,0.484952,1.230,-0.745048
1,S>IO (erro),0.002000,0.402000,0.234,0.168000
2,P(IO),0.486445,0.303026,NaN,NaN


In [5]:
distribution = pd.DataFrame(
    {
        "baseline": baseline_ld.numpy(),
        "adversarial": adversarial_ld.numpy(),
    }
).melt(var_name="condição", value_name="IO−S")
fig_distribution = px.histogram(
    distribution,
    x="IO−S",
    color="condição",
    barmode="overlay",
    opacity=0.60,
    marginal="box",
    title="Distribuição pareada de IO−S",
)
fig_distribution.add_vline(x=0, line_dash="dash", line_color="black")
fig_distribution.show()

deltas = pd.DataFrame(
    {
        "delta IO−S": (adversarial_ld - baseline_ld).numpy(),
        "delta P(IO)": (
            adversarial_io_probability - baseline_io_probability
        ).numpy(),
    }
)
fig_delta = px.scatter(
    deltas,
    x="delta IO−S",
    y="delta P(IO)",
    marginal_x="histogram",
    marginal_y="histogram",
    title="Efeito adversarial pareado por exemplo",
)
fig_delta.add_vline(x=0, line_dash="dash", line_color="black")
fig_delta.add_hline(y=0, line_dash="dash", line_color="black")
fig_delta.show()

## Mecanismo previsto e limite causal

A hipótese do circuito prevê que as duas novas ocorrências do IO ativem a heurística de repetição dos heads **Duplicate Token**. O sinal chega aos heads de **S-Inhibition** como se o IO fosse o nome repetido que deve ser inibido; isso reduz o encaminhamento do IO pelos Name Movers e aumenta a chance de S vencer. O efeito esperado é queda de IO−S, queda de P(IO) e aumento da taxa S>IO.

Esse resultado comportamental sozinho **não prova o circuito**. Ele confirma uma previsão do mecanismo, mas também pode ser produzido por outras rotas do modelo ou por mudanças distribucionais causadas pela inserção. Demonstrar mediação exigiria intervenções causais ou medidas de atenção/ativação nos heads relevantes. A tarefa atual mantém essa análise de heads como opcional e não a usa para ampliar a conclusão.

In [6]:
source_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=PROJECT_ROOT,
    text=True,
).strip()
git_dirty = bool(
    subprocess.check_output(
        ["git", "status", "--porcelain"],
        cwd=PROJECT_ROOT,
        text=True,
    ).strip()
)
assert not git_dirty, "execute from the committed source tree for provenance"

source_paths = {
    "paper/adversarial.py": PROJECT_ROOT / "src/ioi_replication/paper/adversarial.py",
    "paper/dataset.py": PROJECT_ROOT / "src/ioi_replication/paper/dataset.py",
    "paper/metrics.py": PROJECT_ROOT / "src/ioi_replication/paper/metrics.py",
    "runtime.py": PROJECT_ROOT / "src/ioi_replication/runtime.py",
    "notebook": PROJECT_ROOT / "notebooks/10_adversarial_examples.ipynb",
}
source_sha256 = {
    name: hashlib.sha256(path.read_bytes()).hexdigest()
    for name, path in source_paths.items()
}
model_dtype = str(next(model.parameters()).dtype).removeprefix("torch.")
gpu = (
    torch.cuda.get_device_name(device)
    if device.type == "cuda"
    else None
)

result = {
    "paper": "https://arxiv.org/abs/2211.00593",
    "config": {
        "seed": SEED,
        "n": N,
        "batch_size": BATCH_SIZE,
        "model_name": MODEL_NAME,
        "device": str(device),
        "gpu": gpu,
        "dtype": model_dtype,
        "dataset_sha256": dataset.dataset_sha256,
        "prepend_bos": dataset.prepend_bos,
        "padding_side": model.tokenizer.padding_side,
    },
    "metrics": metrics_json,
    "paper_targets": PAPER_ADVERSARIAL_TARGETS,
    "observed_minus_paper": {
        "adversarial_logit_difference": (
            metrics.logit_difference.adversarial.mean
            - PAPER_ADVERSARIAL_TARGETS["adversarial_logit_difference"]
        ),
        "adversarial_s_wins_error_rate": (
            metrics.s_wins_error_rate.adversarial.mean
            - PAPER_ADVERSARIAL_TARGETS[
                "adversarial_s_wins_error_rate"
            ]
        ),
    },
    "methodology": {
        "comparison_scope": (
            "Same five clauses, ordered-pair sampling, middle insertion, and "
            "behavioral metrics as the historical experiment."
        ),
        "differences": [
            "Attack-template sampling is seeded locally; advex.py used the global random module without recording a seed.",
            "Base prompts use this project's deterministic paper dataset generator rather than the historical global RNG state.",
            "TransformerLens 4.3 compatibility mode replaces the historical EasyTransformer runtime.",
            "Confidence intervals use normal 1.96×SE intervals over paired per-example values.",
        ],
        "causal_limit": (
            "Behavioral degradation supports the circuit prediction but does "
            "not by itself establish mediation by Duplicate Token and "
            "S-Inhibition heads."
        ),
    },
    "traceability": {
        "source_commit": source_commit,
        "git_dirty": git_dirty,
        "historical_source_commit": EASY_TRANSFORMER_SOURCE_COMMIT,
        "historical_source_url": EASY_TRANSFORMER_ADVEX_URL,
        "source_sha256": source_sha256,
        "timestamp_utc": datetime.now(timezone.utc).isoformat().replace(
            "+00:00", "Z"
        ),
        "versions": {
            "torch": torch.__version__,
            "transformer_lens": version("transformer-lens"),
            "pandas": version("pandas"),
            "plotly": version("plotly"),
        },
    },
}
output_path = PROJECT_ROOT / "results/paper/10_adversarial_examples.json"
output_path.write_text(
    json.dumps(result, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
print("Resultado salvo em", output_path.relative_to(PROJECT_ROOT))
result

Resultado salvo em results\paper\10_adversarial_examples.json


{'paper': 'https://arxiv.org/abs/2211.00593',
 'config': {'seed': 510,
  'n': 500,
  'batch_size': 16,
  'model_name': 'gpt2',
  'device': 'cuda',
  'gpu': 'NVIDIA GeForce RTX 3050',
  'dtype': 'float32',
  'dataset_sha256': 'c52bf8d46bf8c9774fe8c0eecc968a4cc4d3dc7707b9ff2601272fdd196a1122',
  'prepend_bos': False,
  'padding_side': 'right'},
 'metrics': {'logit_difference_io_minus_s': {'baseline': {'mean': 3.4282398223876953,
    'standard_error': 0.06758862608509969,
    'ci95': [3.2957685494963553, 3.5607110952790353]},
   'adversarial': {'mean': 0.4849517345428467,
    'standard_error': 0.06964455596264552,
    'ci95': [0.34845091313677723, 0.6214525559489161]},
   'delta': {'mean': -2.9432880878448486,
    'standard_error': 0.07171140414086272,
    'ci95': [-3.083839857241736, -2.8027363184479612]}},
  's_wins_error_rate': {'baseline': {'mean': 0.0020000000949949026,
    'standard_error': 0.0020000000760124247,
    'ci95': [-0.0019199280230668206, 0.005919928213056626]},
   'adver